# Tiny ImageNet splits: 10 classes + other

Writes frozen manifests for an 11-way classification task:

- source classes `000`–`009` → labels `0`–`9`
- every other source class (`010`–`199`) → label `10` (`other`)

The downloaded `train/` folder is split **70/30 into train/val per source class** with a
fixed seed, so each of the 200 source classes (including every one folded into
`other`) appears in both in the same proportion. The downloaded `test/` folder (Tiny
ImageNet's official validation set) becomes the `test` manifest and is kept for final
reporting only (PLAN.md D14).

Output, per PLAN.md §3.5: `{train,val,test}.csv` with columns `sample_id,path,label`
(`path` relative to the dataset root), `class_to_idx.json`, and `split.json` recording
how the split was made. An existing split directory is never overwritten.

In [5]:
import csv
import json
import random
from collections import Counter
from pathlib import Path

ROOT = Path("../data/tiny-imagenet")
SEED = 42
TRAIN_FRACTION = 0.7
KEEP = [f"{i:03d}" for i in range(10)]
OTHER = "other"

pct = round(100 * TRAIN_FRACTION)
OUT = ROOT / "splits" / f"top{len(KEEP)}-other_{pct}-{100 - pct}_seed{SEED}"

class_to_idx = {c: i for i, c in enumerate(KEEP)} | {OTHER: len(KEEP)}


def label(source: str) -> int:
    return class_to_idx.get(source, class_to_idx[OTHER])


def images(split: str) -> dict[str, list[Path]]:
    return {
        d.name: sorted(f.relative_to(ROOT) for f in d.iterdir() if f.is_file())
        for d in sorted((ROOT / split).iterdir())
        if d.is_dir()
    }

## Split

Files are sorted before shuffling, and classes are visited in sorted order, so the split
depends only on `SEED` and the files present, not on filesystem listing order.

In [6]:
rng = random.Random(SEED)
splits = {"train": [], "val": [], "test": []}
for files in images("train").values():
    files = rng.sample(files, len(files))
    k = round(TRAIN_FRACTION * len(files))
    splits["train"] += files[:k]
    splits["val"] += files[k:]
for files in images("test").values():
    splits["test"] += files


def row(p: Path) -> tuple[str, str, int]:
    return p.with_suffix("").as_posix(), p.as_posix(), label(p.parent.name)


manifests = {name: [row(p) for p in paths] for name, paths in splits.items()}
{name: len(rows) for name, rows in manifests.items()}

{'train': 70000, 'val': 30000, 'test': 10000}

## Write

In [7]:
OUT.mkdir(parents=True, exist_ok=False)  # refuse to overwrite a frozen split

for name, rows in manifests.items():
    with (OUT / f"{name}.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["sample_id", "path", "label"])
        writer.writerows(rows)

(OUT / "class_to_idx.json").write_text(json.dumps(class_to_idx, indent=2) + "\n")
(OUT / "split.json").write_text(
    json.dumps(
        {
            "root": ROOT.resolve().name,
            "seed": SEED,
            "train_fraction": TRAIN_FRACTION,
            "holdout": "per source class, from train/",
            "test": "test/ (official Tiny ImageNet validation set)",
            "keep": KEEP,
            "other": OTHER,
            "sizes": {name: len(rows) for name, rows in manifests.items()},
        },
        indent=2,
    )
    + "\n"
)
sorted(p.name for p in OUT.iterdir())

FileExistsError: [Errno 17] File exists: '../data/tiny-imagenet/splits/top10-other_70-30_seed42'

## Check

Read back what was written: splits are disjoint, every source class is split 70/30,
every label is in `[0, 11)`.

In [ ]:
def read(name: str) -> list[dict]:
    with (OUT / f"{name}.csv").open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


written = {name: read(name) for name in manifests}
assert json.loads((OUT / "class_to_idx.json").read_text()) == class_to_idx

every = [r for rows in written.values() for r in rows]
assert len({r["sample_id"] for r in every}) == len(every), "sample in two splits"
assert all((ROOT / r["path"]).is_file() for r in every)
assert all(0 <= int(r["label"]) < len(class_to_idx) for r in every)

per_source = {
    name: Counter(Path(r["path"]).parent.name for r in written[name])
    for name in ("train", "val")
}
for source, n in images("train").items():
    k = round(TRAIN_FRACTION * len(n))
    assert (per_source["train"][source], per_source["val"][source]) == (k, len(n) - k)

idx_to_class = {i: c for c, i in class_to_idx.items()}
counts = {n: Counter(int(r["label"]) for r in rows) for n, rows in written.items()}
print(f"{'label':>12}" + "".join(f"{n:>8}" for n in counts))
for i in range(len(class_to_idx)):
    print(
        f"{i:>3} {idx_to_class[i]:>8}" + "".join(f"{c[i]:>8}" for c in counts.values())
    )

## Class balance

`other` is 190 of the 200 source classes, so it is 95% of every split (66,500 train
images against 350 for each kept class). The manifests keep every image, so the split
stays a fixed record; balance is handled when training and evaluating:

- **train:** a `WeightedRandomSampler` with inverse-frequency weights, or class weights
  in the cross-entropy loss;
- **evaluate:** macro-averaged metrics (balanced accuracy, macro F1, per-class recall).
  Plain accuracy is 95% for a model that always predicts `other`.

## Load

The manifests feed a plain torch `Dataset` that yields the batch format of PLAN.md
§3.1.

In [ ]:
from torch.utils.data import DataLoader, Dataset
from torchvision.io import ImageReadMode, decode_image


class ManifestDataset(Dataset):
    def __init__(self, root: Path, manifest: Path):
        self.root = root
        with manifest.open(newline="", encoding="utf-8") as f:
            self.rows = list(csv.DictReader(f))

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, i: int) -> dict:
        row = self.rows[i]
        image = decode_image(str(self.root / row["path"]), mode=ImageReadMode.RGB)
        return {
            "inputs": {"image": image.float() / 255},
            "targets": {"class": int(row["label"])},
            "meta": {"sample_id": row["sample_id"]},
        }


loader = DataLoader(ManifestDataset(ROOT, OUT / "train.csv"), batch_size=8)
batch = next(iter(loader))
(
    batch["inputs"]["image"].shape,
    batch["targets"]["class"],
    batch["meta"]["sample_id"][:2],
)